# NeuroJobs: Random Forest + export JSON rapi

Notebook ini mengolah `job_dataset.csv` jadi **3 file** untuk web NeuroJobs:

| File | Isi | Dipakai web untuk |
|---|---|---|
| `rf_model.json` | model Random Forest (pohon-pohonnya) + daftar skill | prediksi posisi langsung di browser |
| `jobs.json` | **semua** lowongan yang sudah dirapikan + ringkasan skill per posisi | halaman Lowongan + skill gap |
| `hasil_rekomendasi.json` | contoh hasil untuk 1 input (semua lowongan sudah diberi skor) | profil awal + bukti hasil Colab = hasil web |

Jalankan sel dari atas ke bawah. Di sel pertama, upload `job_dataset.csv`.

In [ ]:
import re, json, collections, datetime, zipfile, os
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

try:                                    # kalau jalan di Google Colab
    from google.colab import files
    up = files.upload()                 # pilih job_dataset.csv
    CSV = list(up.keys())[0]; IN_COLAB = True
except ImportError:                     # kalau jalan di laptop
    CSV = 'job_dataset.csv'; IN_COLAB = False

OUT = 'output_neurojobs'; os.makedirs(OUT, exist_ok=True)
raw = pd.read_csv(CSV, encoding='utf-8-sig')
print('Baris:', len(raw), '| kolom:', list(raw.columns))

: 

## 1. Cek kualitas data
Data mentah ini ternyata berantakan. Sel ini menampilkan masalahnya supaya kita tahu **kenapa** perlu dirapikan.

In [ ]:
print('Title kosong       :', raw.Title.isna().sum())
print('JobID dobel        :', raw.JobID.duplicated().sum(), '(mis. JobID "1","2","3" dipakai berkali-kali)')
print('Baris (Title,Skills) sama persis:', raw.duplicated(['Title', 'Skills']).sum())
print('Variasi ExperienceLevel:', sorted(raw.ExperienceLevel.unique()))
print('Contoh YearsOfExperience:', raw.YearsOfExperience.astype(str).unique()[:10])
print('Judul unik         :', raw.Title.nunique(), '-> terlalu banyak untuk jadi label RF')

## 2. Rapikan data
Poin penting: **label RF jangan pakai `Title` mentah.** Ada 219 judul dan 156 di antaranya cuma muncul 1 kali
(`Junior Android Developer`, `Senior Android Developer`, ...). RF tidak bisa belajar dari 1 contoh.
Jadi kata level (Junior/Senior/Lead/Intern/Fresher...) dibuang dari judul, hasilnya jadi kolom `Role`.

Skill juga diseragamkan (`SQL Server basics` -> `sql server`) supaya skill yang sama dihitung sama.

In [ ]:
SEN = r'\b(junior|senior|sr\.?|jr\.?|lead|principal|staff|associate|trainee|intern(ship)?|fresher|entry[- ]level|entry|mid[- ]level|experienced|apprentice|graduate)\b'

def clean_role(t):
    t = re.sub(r'\s*[-–(]\s*(fresher|entry.*|junior|senior|mid.*|experienced|lead|intern.*|trainee)\)?\s*$', '', t, flags=re.I)
    t = re.sub(SEN, '', t, flags=re.I)
    return re.sub(r'\s+', ' ', t).strip(' -–,')

def norm_skill(s):
    s = s.lower().strip()
    s = re.sub(r'\(.*?\)', '', s)
    s = re.sub(r'\b(basics?|fundamentals?|knowledge|familiarity|understanding|exposure|proficiency|experience|skills?)\b( of| with| in)?', '', s)
    return re.sub(r'\s+', ' ', s).strip(' -–')

def level_group(x):
    x = str(x).lower()
    if any(k in x for k in ['senior', 'lead', 'principal']) and 'mid' not in x: return 'Senior'
    if 'mid' in x: return 'Mid-Level'
    if 'experienced' in x: return 'Experienced'
    return 'Entry/Fresher'              # fresher, entry-level, junior

def clean_years(x):
    x = str(x).replace('–', '-').lower()
    return re.sub(r'\s*(years?|yrs?)', '', x).strip()

def split_list(x):
    return [s.strip() for s in str(x).split(';') if s.strip()]

df = raw.copy()
dibuang = df[df.Title.isna() | (df.Title.astype(str).str.strip() == '')]
df = df.drop(dibuang.index).reset_index(drop=True)
print('Dibuang karena Title kosong:', len(dibuang), 'baris', list(dibuang.JobID))

df['Role']   = df.Title.str.strip().map(clean_role)
df['Level']  = df.ExperienceLevel.map(level_group)
df['Years']  = df.YearsOfExperience.map(clean_years)
df['SkillList'] = df.Skills.map(split_list)
df['SkillNorm'] = df.SkillList.map(lambda L: [norm_skill(s) for s in L])   # sejajar dengan SkillList
df['JobKey'] = ['J%04d' % (i + 1) for i in range(len(df))]                 # ID baru yang pasti unik

print('Judul unik:', df.Title.nunique(), '->  Role unik:', df.Role.nunique())
print(df.Role.value_counts().head(8).to_dict())

## 3. Siapkan fitur dan data latih
- **Fitur (X)**: tiap skill = 1 kolom, nilainya 1 kalau lowongan memuat skill itu, 0 kalau tidak. Cuma skill yang muncul minimal 2x yang dipakai.
- **Label (y)**: `Role`. Role yang barisnya kurang dari 5 tidak dilatih (terlalu sedikit contoh), tapi lowongannya **tetap tampil di web**.
- Baris dengan Role + Skills kembar dibuang **khusus untuk latihan**, supaya hasil evaluasi tidak "nyontek" dirinya sendiri.

In [ ]:
MIN_PER_ROLE = 5
train_pool = df.drop_duplicates(['Role', 'Skills'])
vc = train_pool.Role.value_counts()
trained_roles = set(vc[vc >= MIN_PER_ROLE].index)
tr = train_pool[train_pool.Role.isin(trained_roles)].reset_index(drop=True)

cnt = collections.Counter(s for L in tr.SkillNorm for s in set(L) if s)
vocab = sorted(s for s, c in cnt.items() if c >= 2)
vidx = {s: i for i, s in enumerate(vocab)}

def to_vec(norm_skills):
    v = np.zeros(len(vocab), dtype=np.uint8)
    for s in norm_skills:
        if s in vidx: v[vidx[s]] = 1
    return v

X = np.vstack([to_vec(L) for L in tr.SkillNorm])
y = tr.Role.values
print(f'Data latih: {len(tr)} baris | {len(trained_roles)} role | {len(vocab)} fitur skill')

## 4. Latih Random Forest dan ukur akurasinya
Akurasi diukur dengan *cross-validation* (data dibagi 3, bergantian jadi data uji), bukan dari data yang dipakai latihan.

Catatan jujur: dataset ini kelihatannya dibuat dari template (satu role punya ~20 lowongan yang mirip), jadi angka di dunia nyata
biasanya lebih rendah dari angka di sini.

In [ ]:
N_TREES = 100
rf = RandomForestClassifier(n_estimators=N_TREES, random_state=42, n_jobs=-1)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
P = cross_val_predict(rf, X, y, cv=skf, method='predict_proba')
classes_cv = np.array(sorted(set(y)))
order = np.argsort(-P, axis=1)
top1 = np.mean(classes_cv[order[:, 0]] == y)
top3 = np.mean([y[i] in classes_cv[order[i, :3]] for i in range(len(y))])
print(f'Akurasi cross-validation  top-1: {top1:.1%} | top-3: {top3:.1%}')

rf.fit(X, y)                       # model final dilatih dari semua data latih
classes = [str(c) for c in rf.classes_]
imp = sorted(zip(vocab, rf.feature_importances_), key=lambda t: -t[1])[:10]
print('10 skill paling berpengaruh:', [f'{s} ({v:.3f})' for s, v in imp])

## 5. Export model RF ke JSON
Web tidak bisa menjalankan Python, jadi **pohon-pohon RF diekspor apa adanya** dan web menelusurinya sendiri.
Tiap pohon disimpan sebagai array: `f` (skill yang dicek), `l`/`r` (anak kiri = skill tidak ada, kanan = ada), `v` (isi daun).

In [ ]:
def export_tree(est):
    t = est.tree_
    f, l, r, v = [], [], [], []
    for n in range(t.node_count):
        if t.children_left[n] == -1:                      # daun
            f.append(-1); l.append(-1); r.append(-1)
            p = t.value[n][0]; p = p / p.sum()
            nz = np.nonzero(p)[0]
            v.append(int(nz[0]) if len(nz) == 1 else [[int(c), round(float(p[c]), 4)] for c in nz])
        else:
            f.append(int(t.feature[n])); l.append(int(t.children_left[n])); r.append(int(t.children_right[n])); v.append(None)
    return {'f': f, 'l': l, 'r': r, 'v': v}

# contoh pasangan teks mentah -> teks rapi, dipakai untuk memastikan normalisasi di web sama dengan di sini
sample_raw = sorted({s for L in df.SkillList for s in L})
rng = np.random.RandomState(0)
norm_tests = [[s, norm_skill(s)] for s in rng.choice(sample_raw, 60, replace=False)]

model_json = {
    'versi': 1,
    'dibuat': datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
    'algoritma': 'RandomForestClassifier',
    'n_pohon': N_TREES,
    'metrik': {'cv_top1_persen': round(top1 * 100, 1), 'cv_top3_persen': round(top3 * 100, 1),
               'baris_latih': int(len(tr)), 'jumlah_role': len(classes), 'jumlah_skill': len(vocab)},
    'classes': classes,
    'vocab': vocab,
    'norm_tests': norm_tests,
    'trees': [export_tree(e) for e in rf.estimators_],
}
with open(f'{OUT}/rf_model.json', 'w', encoding='utf-8') as fp:
    json.dump(model_json, fp, ensure_ascii=False, separators=(',', ':'))
print('rf_model.json:', round(os.path.getsize(f'{OUT}/rf_model.json') / 1e6, 2), 'MB')

## 6. Export SEMUA lowongan + ringkasan skill per posisi
Semua baris yang sudah dirapikan masuk ke `jobs.json`, termasuk yang role-nya tidak ikut dilatih.
`ringkasan_posisi` berisi skill yang paling sering diminta tiap posisi, dipakai web untuk fitur "skill yang perlu dipelajari".

In [ ]:
def top_skills(sub, n=10):
    c = collections.Counter(); disp = collections.defaultdict(collections.Counter)
    for L, N in zip(sub.SkillList, sub.SkillNorm):
        for d, k in set(zip(L, N)):
            if k: c[k] += 1; disp[k][d] += 1
    return [{'n': k, 'd': disp[k].most_common(1)[0][0], 'persen': round(100 * v / len(sub))} for k, v in c.most_common(n)]

ringkasan = {role: {'jumlah': int(len(g)), 'dilatih': role in set(classes), 'skill_inti': top_skills(g)}
             for role, g in df.groupby('Role')}

jobs = []
for _, r in df.iterrows():
    jobs.append({
        'id': r.JobKey, 'kode_asal': str(r.JobID), 'title': r.Title.strip(), 'role': r.Role,
        'level': r.Level, 'level_asal': r.ExperienceLevel, 'pengalaman': r.Years,
        'skills': r.SkillList, 'skills_norm': r.SkillNorm,
        'tugas': split_list(r.Responsibilities), 'keywords': split_list(r.Keywords),
        'dilatih': r.Role in set(classes),
    })

jobs_json = {'versi': 1, 'jumlah': len(jobs), 'jobs': jobs, 'ringkasan_posisi': ringkasan}
with open(f'{OUT}/jobs.json', 'w', encoding='utf-8') as fp:
    json.dump(jobs_json, fp, ensure_ascii=False, separators=(',', ':'))
print('jobs.json:', len(jobs), 'lowongan,', round(os.path.getsize(f'{OUT}/jobs.json') / 1e6, 2), 'MB')

## 7. Hasil untuk satu contoh input (`hasil_rekomendasi.json`)
Ganti `INPUT_SKILLS` dengan skill yang mau dicoba. File ini dipakai web sebagai **profil awal**,
dan juga bukti: hasil web (RF dijalankan di browser) harus sama dengan hasil Colab ini.

Skor lowongan = *cosine similarity* antara skill kamu dan skill lowongan. RF dipakai untuk memprediksi **posisi**.

In [ ]:
INPUT_SKILLS = ['Python', 'SQL', 'Data Analysis', 'Tableau']

u_norm = {norm_skill(s) for s in INPUT_SKILLS if norm_skill(s)}
known = sorted(s for s in u_norm if s in vidx)
proba = rf.predict_proba(to_vec(u_norm).reshape(1, -1))[0]
rank = np.argsort(-proba)
top_posisi = []
for i in rank[:5]:
    if proba[i] <= 0: continue
    role = classes[i]
    inti = ringkasan[role]['skill_inti']
    top_posisi.append({'posisi': role, 'keyakinan_persen': round(float(proba[i]) * 100, 2),
                       'jumlah_lowongan': ringkasan[role]['jumlah'],
                       'skill_inti_belum_dimiliki': [s['d'] for s in inti if s['n'] not in u_norm][:6]})

rek = []
for j in jobs:
    jn = {k for k in j['skills_norm'] if k}
    hit = u_norm & jn
    cos = len(hit) / np.sqrt(max(len(u_norm), 1) * max(len(jn), 1))
    rek.append({'job_id': j['id'], 'kode_asal': j['kode_asal'], 'posisi': j['title'], 'role': j['role'], 'level': j['level'],
                'pengalaman': j['pengalaman'], 'skor_kecocokan': round(float(cos) * 100, 2),
                'skill_cocok': [d for d, k in zip(j['skills'], j['skills_norm']) if k in hit],
                'skill_kurang': [d for d, k in zip(j['skills'], j['skills_norm']) if k and k not in hit][:6]})
rek.sort(key=lambda r: (-r['skor_kecocokan'], r['job_id']))
for n, r in enumerate(rek, 1): r['peringkat'] = n

hasil = {'status': 'success', 'versi': 1, 'input_skills': INPUT_SKILLS, 'skill_dikenali_model': known,
         'prediksi_utama': {'posisi': top_posisi[0]['posisi'], 'keyakinan_persen': top_posisi[0]['keyakinan_persen']} if top_posisi else None,
         'top_posisi': top_posisi, 'jumlah_lowongan': len(rek), 'rekomendasi_lowongan': rek}
with open(f'{OUT}/hasil_rekomendasi.json', 'w', encoding='utf-8') as fp:
    json.dump(hasil, fp, ensure_ascii=False, indent=1)

print('Skill dikenali model :', known, '| diabaikan:', sorted(u_norm - set(known)))
for t in top_posisi: print(f"  {t['keyakinan_persen']:5.1f}%  {t['posisi']}")
print('Lowongan teratas     :', [(r['kode_asal'], r['posisi'], r['skor_kecocokan']) for r in rek[:3]])

## 8. Unduh hasilnya
Di web NeuroJobs: `rf_model.json` dan `jobs.json` taruh di folder `data/`, `hasil_rekomendasi.json` taruh di folder utama (timpa yang lama).

In [ ]:
zip_name = 'neurojobs_data.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
    for n in ['rf_model.json', 'jobs.json', 'hasil_rekomendasi.json']:
        z.write(f'{OUT}/{n}', n)
print('Dibuat:', zip_name)
if IN_COLAB: files.download(zip_name)